# Task 1.1 — Linear Regression with Gradient Descent

Uses NumPy and pandas only (no scikit-learn). Keep `train.csv` and `test.csv` in the same folder as this notebook.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training shape:", train.shape)
print("Test shape:", test.shape)
print("Columns:", train.columns.tolist())

## 2. Select features and target

Predict `close` using numeric columns other than `close` and `index`. Non-numeric columns (such as dates or symbols) are not used in this simple model.

In [ ]:
target = "close"
if target not in train.columns or target not in test.columns:
    raise ValueError("Both files must contain a close column.")

cols = train.select_dtypes(include=np.number).columns.tolist()
cols = [c for c in cols if c not in [target, "index"] and c in test.columns]

train = train.dropna(subset=[target]).copy()
test = test.dropna(subset=[target]).copy()
X = train[cols].copy()
Xt = test[cols].copy()
y = train[target].to_numpy(dtype=float)
yt = test[target].to_numpy(dtype=float)

print("Features:", cols)
print("Number of features:", len(cols))

## 3. Missing values

Fill missing feature values with the training-set column means, and use the same means for the test set.

In [ ]:
means = X.mean()
X = X.fillna(means)
Xt = Xt.fillna(means)

# Drop columns that were completely missing in training
keep = means[means.notna()].index.tolist()
X = X[keep]
Xt = Xt[keep]
cols = keep

print("Missing training feature values:", int(X.isna().sum().sum()))
print("Missing test feature values:", int(Xt.isna().sum().sum()))

## 4. Outlier treatment (IQR clipping)

For each feature, calculate Q1, Q3 and IQR. Values outside `Q1 - 1.5*IQR` and `Q3 + 1.5*IQR` are clipped to the nearest limit. Limits come from training data and are also applied to test data.

In [ ]:
q1 = X.quantile(0.25)
q3 = X.quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

X = X.clip(lower=low, upper=high, axis=1)
Xt = Xt.clip(lower=low, upper=high, axis=1)
print("Outlier clipping applied to", len(cols), "features.")

## 5. Mean normalization

Formula: `(value - training mean) / (training max - training min)`. Training statistics are reused for test data.

In [ ]:
mean_x = X.mean()
range_x = X.max() - X.min()
range_x = range_x.replace(0, 1)  # avoid division by zero

X = ((X - mean_x) / range_x).to_numpy(dtype=float)
Xt = ((Xt - mean_x) / range_x).to_numpy(dtype=float)
print("X shape:", X.shape)
print("Xt shape:", Xt.shape)

## 6. Train using gradient descent

In [ ]:
m = len(X)
w = np.zeros(X.shape[1])
b = 0.0
alpha = 0.01
costs = []

for i in range(1000):
    pred = np.dot(X, w) + b
    error = pred - y
    cost = np.sum(error ** 2) / (2 * m)
    costs.append(cost)

    dw = np.dot(X.T, error) / m
    db = np.sum(error) / m
    w = w - alpha * dw
    b = b - alpha * db

    if (i + 1) % 100 == 0:
        alpha = alpha * 0.9
        print("Iteration:", i + 1, "Cost:", cost, "Learning rate:", alpha)

## 7. Predict test values

In [ ]:
pred_test = np.dot(Xt, w) + b
print("First 10 actual close values:", yt[:10])
print("First 10 predicted close values:", pred_test[:10])

## 8. Evaluate predictions

R² is a regression score, not literal percentage accuracy. MAPE is the average absolute percentage error. `100 - MAPE` is only an approximate score, not the percentage of predictions that are correct.

In [ ]:
# R-squared
ss_res = np.sum((yt - pred_test) ** 2)
ss_tot = np.sum((yt - np.mean(yt)) ** 2)
r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

# Mean absolute error
mae = np.mean(np.abs(yt - pred_test))

# MAPE: skip zero actual values
nonzero = yt != 0
if np.any(nonzero):
    mape = np.mean(np.abs((yt[nonzero] - pred_test[nonzero]) / yt[nonzero])) * 100
    approx_score = 100 - mape
else:
    mape = np.nan
    approx_score = np.nan

print("R2:", r2)
print("R2 percentage-style:", r2 * 100, "%")
print("MAE:", mae)
print("MAPE:", mape, "%")
print("Approx. score (100 - MAPE):", approx_score, "%")

## 9. Save predictions

In [ ]:
result = pd.DataFrame({"actual_close": yt, "predicted_close": pred_test})
result.to_csv("predictions.csv", index=False)
print("Saved predictions to predictions.csv")
result.head(10)